# Sesión 2 · De Hadoop a Spark
**Profundización I: Procesamiento de Datos — Especialización en Big Data (UCompensar)**

Este notebook acompaña el material HTML de la sesión. Cada celda lleva el mismo
identificador que el bloque de código del material, así que puedes seguir la clase
celda a celda.

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden**. Hadoop y Spark guardan estado entre celdas.
3. Si Colab se reinicia (por ejemplo, tras un rato sin uso), usa
   **Entorno de ejecución → Ejecutar anteriores** desde la celda en la que ibas.
4. Los datos son de *Tiendas Guadua S.A.S.*, una empresa **ficticia**.

In [ ]:
#@title ▶ Crea los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Tiendas Guadua S.A.S.» (empresa FICTICIA).

Genera los archivos con los que se trabaja durante el curso:

    datos/productos.csv    catálogo de 60 productos
    datos/ventas.csv       ~3 000 ventas del primer semestre de 2025, SUCIAS A PROPÓSITO
    datos/resenas.csv      40 reseñas cortas de clientes (texto libre)
    datos/logs_web.jsonl   eventos de navegación de la tienda en línea (JSON por línea)

Solo usa la biblioteca estándar y una semilla fija: en Colab, en Windows o en
macOS se generan exactamente los mismos archivos. Ninguna cifra del material se
escribe a mano: todas salen de ejecutar el código sobre estos datos.

Uso:  python genera_datos.py [carpeta_destino]      (por defecto ./datos)
"""

import csv
import json
import os
import random
import sys
from datetime import date, datetime, timedelta

SEMILLA = 2026

PRODUCTOS = [
    # (nombre, categoría, precio de lista en pesos)
    ("Arroz blanco 1 kg", "Despensa", 4900),
    ("Lentejas 500 g", "Despensa", 3900),
    ("Fríjol cargamanto 500 g", "Despensa", 6800),
    ("Aceite vegetal 1 L", "Despensa", 11900),
    ("Azúcar 1 kg", "Despensa", 4600),
    ("Sal refinada 1 kg", "Despensa", 1900),
    ("Pasta spaghetti 500 g", "Despensa", 3400),
    ("Harina de maíz 1 kg", "Despensa", 3800),
    ("Atún en lata 170 g", "Despensa", 6900),
    ("Panela 500 g", "Despensa", 3200),
    ("Café molido 500 g", "Despensa", 16900),
    ("Chocolate de mesa 250 g", "Despensa", 7500),
    ("Agua sin gas 600 ml", "Bebidas", 1800),
    ("Gaseosa 1,5 L", "Bebidas", 5200),
    ("Jugo de naranja 1 L", "Bebidas", 6400),
    ("Té helado 500 ml", "Bebidas", 3100),
    ("Bebida energizante 250 ml", "Bebidas", 4900),
    ("Leche entera 1 L", "Bebidas", 4300),
    ("Yogur bebible 1 L", "Bebidas", 8200),
    ("Kumis 1 L", "Bebidas", 7400),
    ("Malta 330 ml", "Bebidas", 2500),
    ("Agua con gas 600 ml", "Bebidas", 2100),
    ("Huevos x 30", "Frescos", 17900),
    ("Pollo entero kg", "Frescos", 13900),
    ("Carne molida 500 g", "Frescos", 12400),
    ("Queso campesino 500 g", "Frescos", 11800),
    ("Plátano verde kg", "Frescos", 3600),
    ("Papa pastusa kg", "Frescos", 2900),
    ("Tomate chonto kg", "Frescos", 4200),
    ("Cebolla cabezona kg", "Frescos", 3300),
    ("Aguacate hass unidad", "Frescos", 3900),
    ("Banano kg", "Frescos", 2800),
    ("Detergente en polvo 1 kg", "Aseo", 14500),
    ("Jabón de loza 500 g", "Aseo", 5900),
    ("Limpiador multiusos 1 L", "Aseo", 7800),
    ("Cloro 1 L", "Aseo", 4100),
    ("Papel higiénico x 12", "Aseo", 21900),
    ("Toallas de cocina x 3", "Aseo", 9600),
    ("Suavizante 1 L", "Aseo", 10900),
    ("Esponjas x 3", "Aseo", 3700),
    ("Bolsas de basura x 10", "Aseo", 6200),
    ("Desinfectante 500 ml", "Aseo", 8900),
    ("Champú 400 ml", "Cuidado personal", 15900),
    ("Crema dental 100 ml", "Cuidado personal", 6300),
    ("Jabón de baño x 3", "Cuidado personal", 8400),
    ("Desodorante 150 ml", "Cuidado personal", 13200),
    ("Cepillo dental", "Cuidado personal", 4800),
    ("Acondicionador 400 ml", "Cuidado personal", 16400),
    ("Protector solar 120 ml", "Cuidado personal", 32900),
    ("Toallas higiénicas x 10", "Cuidado personal", 7900),
    ("Crema corporal 400 ml", "Cuidado personal", 18700),
    ("Bombillo LED 9 W", "Hogar", 7900),
    ("Pilas AA x 4", "Hogar", 12900),
    ("Velas x 6", "Hogar", 5400),
    ("Vasos plásticos x 25", "Hogar", 4300),
    ("Platos desechables x 20", "Hogar", 6900),
    ("Encendedor", "Hogar", 2500),
    ("Extensión eléctrica 3 m", "Hogar", 24900),
    ("Olla 24 cm", "Hogar", 59900),
    ("Juego de cubiertos 16 piezas", "Hogar", 39900),
]

PESO_CATEGORIA = {"Despensa": 1.6, "Bebidas": 1.5, "Frescos": 1.4, "Aseo": 1.0,
                  "Cuidado personal": 0.8, "Hogar": 0.5}

CIUDADES = ["Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Cartagena"]
PESO_CIUDAD = [0.34, 0.20, 0.15, 0.12, 0.10, 0.09]

# Formas "sucias" en que llega escrita cada ciudad desde los distintos sistemas
VARIANTES_CIUDAD = {
    "Bogotá": ["bogota", "BOGOTÁ", "Bogotá D.C.", "Bogota", " Bogotá"],
    "Medellín": ["Medellin", "medellín", "MEDELLIN"],
    "Cali": ["cali", "Cali ", "CALI"],
    "Barranquilla": ["barranquilla", "B/quilla", "Barranquilla "],
    "Bucaramanga": ["bucaramanga", "B/manga"],
    "Cartagena": ["cartagena", "Cartagena de Indias"],
}

VARIANTES_CANAL = {"Tienda": ["tienda", "TIENDA"], "Web": ["WEB", "web "], "App": ["app", "APP"]}

PAGO_POR_CANAL = {
    "Tienda": (["Efectivo", "Tarjeta", "Billetera digital"], [0.35, 0.45, 0.20]),
    "Web": (["Tarjeta", "Transferencia", "Billetera digital"], [0.60, 0.30, 0.10]),
    "App": (["Billetera digital", "Tarjeta", "Transferencia"], [0.50, 0.40, 0.10]),
}

RESENAS = [
    # (calificación, texto). Reseñas ficticias escritas para el curso.
    (5, "La entrega fue muy rápida y el pedido llegó completo. Excelente servicio."),
    (4, "Buenos precios y buena calidad, pero la app se demora en cargar el carrito."),
    (2, "El pedido llegó tarde y faltaba un producto. Tuve que llamar dos veces."),
    (5, "Me encanta comprar por la app, es fácil y el domicilio siempre llega a tiempo."),
    (1, "Pésima experiencia: cobraron dos veces el pago y nadie responde en el chat."),
    (4, "Los frescos llegaron en buen estado. El aguacate estaba perfecto."),
    (3, "Precios normales. La tienda es amplia pero las filas en caja son largas."),
    (5, "Excelente atención en la tienda de Medellín, el personal es muy amable."),
    (2, "La página web se cae al momento de pagar. Perdí el carrito dos veces."),
    (4, "Buena variedad de productos de aseo y descuentos interesantes en la app."),
    (1, "Llegó el huevo roto y el pollo sin refrigerar. No vuelvo a pedir frescos."),
    (5, "Rápido, económico y sin complicaciones. Recomendado."),
    (3, "El domicilio llegó bien, pero el empaque venía golpeado."),
    (4, "Me gusta pagar con billetera digital, es rápido. Faltan más promociones."),
    (2, "El precio en la app no coincidía con el precio en la tienda."),
    (5, "Pedí a las ocho de la mañana y a las diez ya tenía todo en casa. Muy bien."),
    (3, "Regular. La calidad del café ha bajado y el precio subió."),
    (4, "La tienda de Cali siempre está limpia y ordenada. Buen surtido."),
    (1, "Cancelaron mi pedido sin avisar y el reembolso tardó una semana."),
    (5, "Excelente calidad en carnes y quesos. El domicilio fue puntual."),
    (2, "La app no me deja aplicar el cupón de descuento. Muy frustrante."),
    (4, "Buena experiencia en general, aunque el pedido llegó un poco tarde."),
    (3, "Productos buenos pero la entrega es lenta los fines de semana."),
    (5, "Siempre encuentro lo que necesito y a buen precio. La mejor opción del barrio."),
    (1, "El servicio al cliente es terrible, nadie resuelve nada por el chat."),
    (4, "La compra por la web fue fácil y el pago con tarjeta funcionó sin problema."),
    (2, "El producto llegó vencido. Revisen las fechas antes de enviar."),
    (5, "Muy buena atención en Barranquilla, rápidos en caja y muy amables."),
    (3, "Normal. Nada especial, pero cumple."),
    (4, "Las ofertas de los martes son muy buenas, compro el mercado completo."),
    (2, "Demasiada demora en el domicilio y el repartidor no encontraba la dirección."),
    (5, "Calidad excelente y precios justos. La app funciona perfecto."),
    (1, "Me llegó un pedido equivocado y no me dejaron devolverlo."),
    (4, "Buen precio en productos de despensa. El arroz y el aceite siempre en oferta."),
    (3, "La tienda de Bucaramanga es pequeña y a veces no hay todo lo del catálogo."),
    (5, "Entrega rápida, todo bien empacado y frío. Muy recomendado."),
    (2, "Cobran el domicilio muy caro para pedidos pequeños."),
    (4, "Me gusta que en la app se ve el estado del pedido en tiempo real."),
    (3, "El pago con transferencia tardó en confirmarse, pero al final llegó todo."),
    (5, "Excelente servicio y excelente calidad. Seguiré comprando en Cartagena."),
]


def _semestre(rng):
    """Días del 1 de enero al 30 de junio de 2025, con más peso el fin de semana
    y una tendencia suave al alza."""
    inicio = date(2025, 1, 1)
    dias = [inicio + timedelta(days=i) for i in range((date(2025, 6, 30) - inicio).days + 1)]
    pesos = [(1.4 if d.weekday() >= 5 else 1.0) * (1 + 0.12 * (d.month - 1) / 5) for d in dias]
    return dias, pesos


def _ventas(rng):
    dias, pesos_dia = _semestre(rng)
    pesos_producto = [PESO_CATEGORIA[c] for _, c, _ in PRODUCTOS]
    clientes = [f"C{i:04d}" for i in range(1, 1501)]
    # Pocos clientes compran mucho: pesos decrecientes
    pesos_cliente = [1 / (1 + i) ** 0.6 for i in range(len(clientes))]

    filas = []
    for dia in sorted(rng.choices(dias, weights=pesos_dia, k=3000)):
        ciudad = rng.choices(CIUDADES, weights=PESO_CIUDAD)[0]
        # La app gana participación mes a mes
        p_app = 0.12 + 0.03 * (dia.month - 1)
        canal = rng.choices(["Tienda", "Web", "App"], weights=[0.63 - p_app, 0.25, p_app + 0.12])[0]
        indice = rng.choices(range(len(PRODUCTOS)), weights=pesos_producto)[0]
        _, _, precio = PRODUCTOS[indice]
        if precio < 10000:
            cantidad = rng.choices([1, 2, 3, 4, 5, 6], weights=[30, 28, 18, 12, 7, 5])[0]
        else:
            cantidad = rng.choices([1, 2, 3], weights=[70, 24, 6])[0]
        if canal == "Tienda":
            descuento = rng.choices([0, 0.05, 0.10], weights=[85, 10, 5])[0]
        else:
            descuento = rng.choices([0, 0.05, 0.10, 0.15], weights=[55, 20, 15, 10])[0]
        medios, pesos_pago = PAGO_POR_CANAL[canal]
        pago = rng.choices(medios, weights=pesos_pago)[0]
        # En tienda se puede comprar sin registrarse: ese nulo NO es un error
        if canal == "Tienda" and rng.random() < 0.40:
            cliente = ""
        else:
            cliente = rng.choices(clientes, weights=pesos_cliente)[0]
        filas.append({
            "fecha": dia.isoformat(),
            "ciudad": ciudad,
            "canal": canal,
            "id_cliente": cliente,
            "id_producto": f"P{indice + 1:03d}",
            "cantidad": cantidad,
            "precio_unitario": precio,
            "descuento": descuento,
            "metodo_pago": pago,
        })

    for i, f in enumerate(filas, start=1):
        f["id_venta"] = f"V{i:05d}"

    # ---- Suciedad controlada: los problemas típicos de un consolidado real ----
    for f in filas:
        r = rng.random()
        if r < 0.07:
            f["ciudad"] = rng.choice(VARIANTES_CIUDAD[f["ciudad"]])
        elif r < 0.075:
            f["ciudad"] = ""                                   # ciudad sin registrar
        if rng.random() < 0.03:
            f["canal"] = rng.choice(VARIANTES_CANAL[f["canal"]])
        if rng.random() < 0.025:
            f["precio_unitario"] = "$" + f"{f['precio_unitario']:,}".replace(",", ".")
        if rng.random() < 0.04:
            d = date.fromisoformat(f["fecha"])
            f["fecha"] = d.strftime("%d/%m/%Y")                # otro formato de fecha
        r = rng.random()
        if r < 0.004:
            f["cantidad"] = -rng.randint(1, 3)                 # cantidad negativa
        elif r < 0.006:
            f["cantidad"] = 0
        elif r < 0.0075:
            f["cantidad"] = f["cantidad"] * 100 + rng.randint(0, 9) * 10   # error de digitación
        if rng.random() < 0.02:
            f["descuento"] = ""
        if rng.random() < 0.01:
            f["metodo_pago"] = ""

    # Doble carga: ~1 % de las ventas aparece repetida justo después del original
    con_duplicados = []
    for f in filas:
        con_duplicados.append(f)
        if rng.random() < 0.01:
            con_duplicados.append(dict(f))
    return con_duplicados


def _resenas(rng):
    dias, _ = _semestre(rng)
    filas = []
    for i, (calificacion, texto) in enumerate(RESENAS, start=1):
        fila = {
            "id_resena": f"R{i:03d}",
            "fecha": rng.choice(dias).isoformat(),
            "ciudad": rng.choices(CIUDADES, weights=PESO_CIUDAD)[0],
            "canal": rng.choices(["Tienda", "Web", "App"], weights=[0.3, 0.3, 0.4])[0],
            "id_producto": f"P{rng.randint(1, len(PRODUCTOS)):03d}",
            "calificacion": calificacion,
            "texto": texto,
        }
        # Lo que dice el texto manda sobre el sorteo (sin más llamadas a rng: ventas y logs no cambian)
        t = texto.lower()
        for ciudad in CIUDADES:
            if ciudad.lower() in t:
                fila["ciudad"] = ciudad
        if " app" in t:
            fila["canal"] = "App"
        elif "web" in t:
            fila["canal"] = "Web"
        elif "tienda de" in t or "en caja" in t or "la tienda es" in t:
            fila["canal"] = "Tienda"
        elif fila["canal"] == "Tienda" and any(p in t for p in ("domicilio", "pedido", "pedí", "entrega",
                                                                "repartidor", "llegó", "envi", "chat")):
            fila["canal"] = "Web"                      # un domicilio o un chat no son de la caja
        filas.append(fila)
    filas.sort(key=lambda f: f["fecha"])
    return filas


def _logs(rng, n=1500):
    """Eventos de la tienda en línea en junio de 2025. Semiestructurados: no
    todos los eventos tienen los mismos campos y algunos van anidados."""
    eventos = []
    inicio = datetime(2025, 6, 1)
    for _ in range(n):
        ts = inicio + timedelta(seconds=rng.randint(0, 30 * 24 * 3600 - 1))
        tipo = rng.choices(["buscar", "ver_producto", "agregar_carrito", "pagar"],
                           weights=[25, 45, 20, 10])[0]
        movil = rng.random() < 0.62
        evento = {
            "ts": ts.isoformat(timespec="seconds"),
            "sesion": f"s-{rng.randint(0, 0xFFFFF):05x}",
            "evento": tipo,
            "dispositivo": {"tipo": "movil" if movil else "escritorio",
                            "so": rng.choice(["android", "ios"]) if movil else rng.choice(["windows", "macos", "linux"])},
            "ciudad": rng.choices(CIUDADES, weights=PESO_CIUDAD)[0],
        }
        if rng.random() < 0.7:
            evento["id_cliente"] = f"C{rng.randint(1, 1500):04d}"
        if tipo == "buscar":
            evento["consulta"] = rng.choice(["arroz", "cafe", "detergente", "leche", "huevos",
                                             "protector solar", "pilas", "pollo", "aceite"])
        else:
            evento["id_producto"] = f"P{rng.randint(1, len(PRODUCTOS)):03d}"
        if tipo == "pagar":
            evento["pago"] = {"medio": rng.choice(["Tarjeta", "Billetera digital", "Transferencia"]),
                              "aprobado": rng.random() < 0.9}
        evento["duracion_ms"] = rng.randint(80, 4000)
        eventos.append(evento)
    eventos.sort(key=lambda e: e["ts"])
    return eventos


def _escribir_csv(ruta, filas, columnas):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=columnas)
        escritor.writeheader()
        escritor.writerows(filas)


def crear_datos(carpeta="datos"):
    """Crea los cuatro archivos del caso en `carpeta` y devuelve sus rutas."""
    os.makedirs(carpeta, exist_ok=True)
    rng = random.Random(SEMILLA)

    productos = [{"id_producto": f"P{i:03d}", "nombre": n, "categoria": c, "precio_lista": p}
                 for i, (n, c, p) in enumerate(PRODUCTOS, start=1)]
    ventas = _ventas(rng)
    resenas = _resenas(rng)
    logs = _logs(rng)

    rutas = {
        "productos": os.path.join(carpeta, "productos.csv"),
        "ventas": os.path.join(carpeta, "ventas.csv"),
        "resenas": os.path.join(carpeta, "resenas.csv"),
        "logs": os.path.join(carpeta, "logs_web.jsonl"),
    }
    _escribir_csv(rutas["productos"], productos, ["id_producto", "nombre", "categoria", "precio_lista"])
    _escribir_csv(rutas["ventas"], ventas,
                  ["id_venta", "fecha", "ciudad", "canal", "id_cliente", "id_producto",
                   "cantidad", "precio_unitario", "descuento", "metodo_pago"])
    _escribir_csv(rutas["resenas"], resenas,
                  ["id_resena", "fecha", "ciudad", "canal", "id_producto", "calificacion", "texto"])
    with open(rutas["logs"], "w", encoding="utf-8") as f:
        for e in logs:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

    print(f"productos.csv : {len(productos):>5} filas")
    print(f"ventas.csv    : {len(ventas):>5} filas")
    print(f"resenas.csv   : {len(resenas):>5} filas")
    print(f"logs_web.jsonl: {len(logs):>5} eventos")
    return rutas

# Crea la carpeta datos/ con los cuatro archivos del caso (los mismos de la sesión 1)
rutas = crear_datos("datos")

In [ ]:
#@title ▶ Trae los resultados de la sesión 1 (ejecútala sin modificar) { display-mode: "form" }
# [sesion1]
# Repite la limpieza de la sesión 1 y deja listos:
#   df                  las ventas limpias en pandas, con total, mes y categoría
#   ventas_limpias.csv  ciudad,total: la entrada del job de Hadoop
import pandas as pd

ventas = pd.read_csv("datos/ventas.csv")
productos = pd.read_csv("datos/productos.csv")

CIUDAD_CANONICA = {
    "bogota": "Bogotá", "bogota d.c.": "Bogotá",
    "medellin": "Medellín",
    "cali": "Cali",
    "barranquilla": "Barranquilla", "b/quilla": "Barranquilla",
    "bucaramanga": "Bucaramanga", "b/manga": "Bucaramanga",
    "cartagena": "Cartagena", "cartagena de indias": "Cartagena",
}


def limpiar_ventas(v):
    """Los seis pasos de limpieza de la sesión 1, en el mismo orden."""
    d = v.drop_duplicates().copy()                                      # unicidad
    norma = (d["ciudad"].str.strip().str.lower().str.normalize("NFKD")
             .str.encode("ascii", errors="ignore").str.decode("ascii"))
    d["ciudad"] = norma.map(CIUDAD_CANONICA).fillna("Sin dato")        # consistencia
    d["canal"] = d["canal"].str.strip().str.capitalize()
    d["precio_unitario"] = pd.to_numeric(                               # validez
        d["precio_unitario"].astype(str).str.replace(r"[$.\s]", "", regex=True), errors="coerce")
    iso = pd.to_datetime(d["fecha"], format="%Y-%m-%d", errors="coerce")
    d["fecha"] = iso.fillna(pd.to_datetime(d["fecha"], format="%d/%m/%Y", errors="coerce"))
    d = d[d["cantidad"].between(1, 50)].copy()                          # exactitud
    d["descuento"] = d["descuento"].fillna(0)                           # completitud
    d["id_cliente"] = d["id_cliente"].fillna("ANONIMO")
    d["metodo_pago"] = d["metodo_pago"].fillna("Sin dato")
    d["total"] = d["cantidad"] * d["precio_unitario"] * (1 - d["descuento"])
    d["mes"] = d["fecha"].dt.strftime("%Y-%m")
    return d.merge(productos[["id_producto", "nombre", "categoria"]],
                   on="id_producto", how="left", validate="many_to_one")


df = limpiar_ventas(ventas)
df[["ciudad", "total"]].to_csv("ventas_limpias.csv", index=False)
print(f"df: {len(df)} ventas limpias · total {df['total'].sum() / 1e6:.1f} millones de pesos")
print("ventas_limpias.csv listo para Hadoop")

## Bloque 1 · Módulo 1 — El entorno de hoy

In [ ]:
# [entorno]
import sys
import pyspark

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("PySpark:", pyspark.__version__)
!java -version 2>&1 | head -n 1

## Bloque 1 · Módulo 2 — Hadoop en Colab

In [ ]:
# [hadoop_descarga]
import os
import platform
import urllib.request

VERSION = "3.5.0"
# Colab corre en procesadores x86_64; para equipos ARM (aarch64) Apache publica otro archivo
sufijo = "-aarch64" if platform.machine() == "aarch64" else ""
ARCHIVO = f"hadoop-{VERSION}{sufijo}.tar.gz"
ESPEJOS = [f"https://dlcdn.apache.org/hadoop/common/hadoop-{VERSION}/",
           f"https://archive.apache.org/dist/hadoop/common/hadoop-{VERSION}/"]


def descargar(nombre):
    """Intenta el espejo principal de Apache y, si falla, su archivo histórico.
    Baja a un archivo «.parcial» y solo al final le pone su nombre: una descarga
    cortada no se confunde con una completa."""
    for base in ESPEJOS:
        try:
            urllib.request.urlretrieve(base + nombre, nombre + ".parcial")
            os.replace(nombre + ".parcial", nombre)
            return
        except OSError as error:
            print("Sin respuesta de", base, "·", error)
    raise RuntimeError(f"No se pudo descargar {nombre}")


for nombre in [ARCHIVO, ARCHIVO + ".sha512"]:
    if not os.path.exists(nombre):
        print("Descargando", nombre, "…")
        descargar(nombre)
print(f"{ARCHIVO}: {os.path.getsize(ARCHIVO) / 1e6:,.0f} MB")

In [ ]:
# [hadoop_integridad]
import hashlib

# CONTROL DE SEGURIDAD: ¿el archivo es exactamente el que publicó Apache?
publicado = open(ARCHIVO + ".sha512").read().split("=")[-1].strip()
huella = hashlib.sha512()
with open(ARCHIVO, "rb") as f:
    for trozo in iter(lambda: f.read(1 << 20), b""):      # de a 1 MB
        huella.update(trozo)
print("SHA-512 publicado:", publicado[:32], "…")
print("SHA-512 calculado:", huella.hexdigest()[:32], "…")
print("¿Íntegro?", huella.hexdigest() == publicado)

In [ ]:
# [hadoop_instala]
import shutil
import subprocess

if not os.path.isdir(f"hadoop-{VERSION}"):
    subprocess.run(["tar", "-xzf", ARCHIVO], check=True)    # descomprimir: unos 30 s

# Variables de entorno que Hadoop necesita; las órdenes «!» y %%bash las heredan
os.environ["JAVA_HOME"] = os.path.dirname(os.path.dirname(os.path.realpath(shutil.which("java"))))
os.environ["HADOOP_HOME"] = os.path.abspath(f"hadoop-{VERSION}")
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["PATH"]
# Colab trabaja como root: Hadoop exige declararlo para arrancar sus servicios
os.environ["HDFS_NAMENODE_USER"] = "root"
os.environ["HDFS_DATANODE_USER"] = "root"

print("JAVA_HOME   =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
!hadoop version | head -n 1

## Bloque 1 · Módulo 3 — HDFS en la práctica

In [ ]:
# [hdfs_config]
# Modo PSEUDODISTRIBUIDO: NameNode y DataNode como servicios separados, en esta máquina
CONF = os.path.join(os.environ["HADOOP_HOME"], "etc", "hadoop")
BASE = os.path.abspath("hdfs")       # carpeta local donde HDFS guarda lo suyo


def escribir_xml(archivo, propiedades):
    """Hadoop se configura con archivos XML de pares nombre → valor."""
    filas = "\n".join(f"  <property><name>{k}</name><value>{v}</value></property>"
                      for k, v in propiedades.items())
    with open(os.path.join(CONF, archivo), "w") as f:
        f.write(f"<configuration>\n{filas}\n</configuration>\n")


escribir_xml("core-site.xml", {"fs.defaultFS": "hdfs://localhost:9000"})
escribir_xml("hdfs-site.xml", {
    "dfs.replication": 1,                             # un solo DataNode → una sola copia
    "dfs.namenode.name.dir": f"file://{BASE}/nombres",
    "dfs.datanode.data.dir": f"file://{BASE}/bloques",
    "dfs.namenode.fs-limits.min-block-size": 65536,   # permite bloques pequeños, solo para aprender
})
!cat $HADOOP_HOME/etc/hadoop/hdfs-site.xml

In [ ]:
%%bash
# [hdfs_arranque]
# 1) Formatear: crea un sistema de archivos HDFS vacío (solo la primera vez)
if [ ! -d hdfs/nombres/current ]; then
  hdfs namenode -format -force -nonInteractive > formato.log 2>&1 && echo "NameNode formateado"
fi
# 2) Arrancar los dos servicios en segundo plano (si no están ya en marcha)
jps | grep -q " NameNode" || hdfs --daemon start namenode
jps | grep -q " DataNode" || hdfs --daemon start datanode
# 3) Esperar a que el DataNode se reporte ante el NameNode
for intento in $(seq 30); do
  hdfs dfsadmin -report 2>/dev/null | grep -q "Live datanodes (1)" && break
  sleep 2
done
hdfs dfsadmin -report 2>/dev/null | grep -E "^(Live datanodes|Name:)"
jps          # procesos de Java en marcha: NameNode y DataNode

In [ ]:
%%bash
# [hdfs_subir]
# Las órdenes de HDFS se parecen a las de Linux: hdfs dfs -mkdir, -put, -ls, -cat…
hdfs dfs -mkdir -p /guadua/crudo /guadua/limpio
hdfs dfs -put -f datos/productos.csv datos/logs_web.jsonl /guadua/crudo/
# ventas.csv con bloques de 64 KB (en un clúster real: 128 MB) para ver cómo se parte
hdfs dfs -D dfs.blocksize=65536 -put -f datos/ventas.csv /guadua/crudo/
hdfs dfs -put -f ventas_limpias.csv /guadua/limpio/
hdfs dfs -du -h /guadua/crudo /guadua/limpio

In [ ]:
%%bash
# [hdfs_bloques]
# Tamaño, tamaño de bloque y factor de replicación que HDFS registró para cada archivo
hdfs dfs -stat "%n | %b bytes | bloque de %o bytes | replicas: %r" \
  /guadua/crudo/ventas.csv /guadua/crudo/productos.csv /guadua/crudo/logs_web.jsonl

In [ ]:
%%bash
# [hdfs_fsck]
# fsck: el NameNode informa dónde está cada bloque del archivo
hdfs fsck /guadua/crudo/ventas.csv -files -blocks -locations 2>/dev/null \
  | grep -E "blk_|Total blocks|Status" | sed -E 's/BP-[^:]+://; s/,DS-[^,]+//'

In [ ]:
%%bash
# [hdfs_permisos]
# CONTROL DE SEGURIDAD: permisos al estilo Linux. Lo crudo solo para el dueño y su grupo
hdfs dfs -chmod -R 750 /guadua/crudo
hdfs dfs -stat "%A  %u:%g  %n" /guadua/crudo /guadua/limpio

In [ ]:
%%bash
# [hdfs_leer]
# Leer de HDFS sin descargar: las primeras líneas y un resumen de la carpeta
hdfs dfs -head /guadua/crudo/ventas.csv | head -n 3
hdfs dfs -count -v -h /guadua

**Ejercicio (módulo 3).** Si Guadua tuviera un clúster de verdad, con bloques de 128 MB y replicación 3,
¿en cuántos bloques quedaría `ventas.csv` y cuántos bytes ocuparía en total en los discos del clúster?
La pista y la solución están en el material HTML.

## Bloque 1 · Módulo 4 — Hadoop Streaming: el mapper y el reducer de la sesión 1

In [ ]:
%%writefile mapper.py
import sys

# MAPPER: lee líneas de la entrada estándar y emite «clave<TAB>valor»
for linea in sys.stdin:
    ciudad, total = linea.strip().split(",")
    if ciudad == "ciudad":        # salta el encabezado
        continue
    print(f"{ciudad}\t{total}")

In [ ]:
%%writefile reducer.py
import sys

# REDUCER: recibe las líneas ORDENADAS por clave y acumula mientras la clave no cambie
actual, suma = None, 0.0
for linea in sys.stdin:
    clave, valor = linea.rstrip("\n").split("\t")
    if clave != actual:
        if actual is not None:
            print(f"{actual}\t{suma:.0f}")
        actual, suma = clave, 0.0
    suma += float(valor)
if actual is not None:
    print(f"{actual}\t{suma:.0f}")

In [ ]:
%%bash
# [streaming_job]
# El job de la sesión 1, ahora en Hadoop. La carpeta de salida no debe existir:
# se borra antes para poder repetir la celda sin error (y sin duplicar resultados)
hdfs dfs -rm -r -f -skipTrash /guadua/resultados/por_ciudad > /dev/null
hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" \
  -reducer "python3 reducer.py" \
  -input /guadua/limpio/ventas_limpias.csv \
  -output /guadua/resultados/por_ciudad 2> job.log || tail -n 20 job.log
# El registro del job es largo (job.log): miremos solo sus contadores principales
sed -n '/mapreduce.Job: Counters/,$p' job.log | grep -E "Map input records|Map output records|Reduce input groups|Reduce output records"
hdfs dfs -ls -C /guadua/resultados/por_ciudad
hdfs dfs -cat /guadua/resultados/por_ciudad/part-00000

In [ ]:
# [streaming_verifica]
import io

# CONTROL DE COHERENCIA: el resultado de Hadoop contra el de pandas
salida = subprocess.run(["hdfs", "dfs", "-cat", "/guadua/resultados/por_ciudad/part-00000"],
                        capture_output=True, text=True, check=True).stdout
hadoop_ciudad = pd.read_csv(io.StringIO(salida), sep="\t", names=["ciudad", "total"],
                            index_col="ciudad")["total"]
comparacion = pd.DataFrame({"Hadoop": hadoop_ciudad,
                            "pandas": df.groupby("ciudad")["total"].sum().round().astype(int)})
comparacion["diferencia"] = comparacion["Hadoop"] - comparacion["pandas"]
comparacion

In [ ]:
# [streaming_partes]
# Cuatro archivos de entrada, como los cuatro bloques de la sesión 1: Hadoop lanza un mapper por cada uno
os.makedirs("partes", exist_ok=True)
tam = -(-len(df) // 4)                                   # división hacia arriba
for i in range(4):
    df[["ciudad", "total"]].iloc[i * tam:(i + 1) * tam].to_csv(f"partes/parte-{i + 1}.csv", index=False)
!hdfs dfs -put -f partes /guadua/limpio/
!hdfs dfs -ls -C /guadua/limpio/partes

In [ ]:
%%bash
# [streaming_combiner]
# El mismo job dos veces, sobre las cuatro partes y con dos reducers: sin combiner y con combiner
ejecutar () {        # $1: nombre del job; lo demás: opciones adicionales de Streaming
  nombre=$1; shift
  hdfs dfs -rm -r -f -skipTrash /guadua/resultados/$nombre > /dev/null
  hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar \
    -D mapreduce.job.reduces=2 \
    -files mapper.py,reducer.py \
    -mapper "python3 mapper.py" "$@" -reducer "python3 reducer.py" \
    -input /guadua/limpio/partes \
    -output /guadua/resultados/$nombre 2> job_$nombre.log || tail -n 20 job_$nombre.log
  echo "== $nombre"
  grep -o "number of splits:[0-9]*" job_$nombre.log
  sed -n '/mapreduce.Job: Counters/,$p' job_$nombre.log \
    | grep -E "Map output records|Combine output records|Reduce input records|Reduce shuffle bytes"
}
ejecutar sin_combiner
ejecutar con_combiner -combiner "python3 reducer.py"

# ¿Dieron lo mismo? Se juntan las partes de cada job, se ordenan y se comparan
if diff <(hdfs dfs -cat /guadua/resultados/sin_combiner/part-* | sort) \
        <(hdfs dfs -cat /guadua/resultados/con_combiner/part-* | sort) > /dev/null
then echo "== mismo resultado con y sin combiner"
else echo "== los resultados difieren"
fi

In [ ]:
%%bash
# [streaming_reducers]
# Con dos reducers hay dos archivos de salida: cada ciudad cae en uno según el hash de su nombre
for parte in part-00000 part-00001; do
  echo "== $parte"
  hdfs dfs -cat /guadua/resultados/con_combiner/$parte
done

**Ejercicio (módulo 4).** Cuenta con Hadoop Streaming **cuántas ventas** hubo en cada ciudad, sobre las
cuatro partes. ¿Puedes reutilizar `reducer.py`? Resuélvelo en una celda nueva (**+ Código**); la pista y la solución están en el material HTML.

## Bloque 1 · Módulo 6 — Spark: el ecosistema

In [ ]:
# [spark_sesion]
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# La SparkSession es la puerta de entrada a Spark. «local[*]»: driver y ejecutor en este proceso, con un hilo de trabajo por núcleo
spark = (SparkSession.builder
         .appName("tiendas-guadua")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "America/Bogota")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")      # el registro interno de Java solo mostrará errores
print("Spark", spark.version, "· núcleos disponibles:", sc.defaultParallelism)

## Bloque 1 · Módulo 7 — Arquitectura y RDD

In [ ]:
# [rdd_particiones]
# Un RDD es una colección repartida en PARTICIONES; cada partición la procesa una tarea
numeros = sc.parallelize(range(1, 11), 4)
print("Particiones:", numeros.getNumPartitions())
print(numeros.glom().collect())          # glom: muestra el contenido de cada partición

In [ ]:
# [rdd_conteo]
import re

# El conteo de palabras de la sesión 1 en Spark: map → shuffle → reduce
textos = pd.read_csv("datos/resenas.csv")["texto"].tolist()
lineas = sc.parallelize(textos, 4)
conteo = (lineas.flatMap(lambda t: re.findall(r"[a-záéíóúüñ]+", t.lower()))   # map: palabras
                .map(lambda palabra: (palabra, 1))                            # pares (clave, 1)
                .reduceByKey(lambda a, b: a + b))                             # shuffle + reduce
print("Palabras distintas:", conteo.count())
conteo.takeOrdered(8, key=lambda par: (-par[1], par[0]))

In [ ]:
# [rdd_verifica]
from collections import Counter

# CONTROL: el resultado de Spark contra un conteo en Python puro
en_python = Counter(p for t in textos for p in re.findall(r"[a-záéíóúüñ]+", t.lower()))
dict(conteo.collect()) == dict(en_python)

## Bloque 2 · Módulo 8 — Transformaciones, acciones y el DAG

In [ ]:
# [perezosa]
# Las TRANSFORMACIONES solo anotan qué hacer: esta línea no lee ni calcula nada
largas = lineas.filter(lambda t: len(t) > 60).map(lambda t: t.upper())
print(type(largas).__name__, "· aún sin ejecutar")

# Las ACCIONES (count, take, collect, save…) disparan el trabajo
print("Reseñas de más de 60 caracteres:", largas.count())
print(largas.take(1))

In [ ]:
# [linaje]
# El linaje: la receta que Spark guarda para (re)calcular el resultado.
# Cada «+-» marca un shuffle, es decir, el límite entre dos ETAPAS
print(conteo.toDebugString().decode())

## Bloque 2 · Módulo 9 — DataFrames y Spark SQL

In [ ]:
# [df_crudo]
# Un DataFrame: datos con ESQUEMA (columnas con nombre y tipo). Spark adivina los tipos…
crudo = spark.read.csv("datos/ventas.csv", header=True, inferSchema=True)
crudo.printSchema()

In [ ]:
# [df_mostrar]
crudo.show(5)

In [ ]:
# [df_canal]
# …y la suciedad sigue ahí: Spark no limpia por nosotros
crudo.groupBy("canal").count().orderBy(F.desc("count"), "canal").show()

In [ ]:
# [df_sql]
# Otra pregunta, ahora en SQL: filas por ciudad, sobre una vista temporal del DataFrame
crudo.createOrReplaceTempView("ventas_crudas")
spark.sql("""
    SELECT ciudad, COUNT(*) AS filas
    FROM ventas_crudas
    GROUP BY ciudad
    ORDER BY filas DESC, ciudad
    LIMIT 8
""").show()

In [ ]:
# [df_hdfs]
# Spark lee directamente de HDFS: el almacenamiento de Hadoop, el procesamiento de Spark
desde_hdfs = spark.read.csv("hdfs://localhost:9000/guadua/crudo/ventas.csv", header=True)
print("Filas leídas desde HDFS:", desde_hdfs.count())
print("Particiones en Spark:", desde_hdfs.rdd.getNumPartitions())

## Bloque 2 · Módulo 10 — Leer con esquema
Declaramos el esquema que deben cumplir las ventas (un contrato) y vemos qué filas no lo cumplen.
Comparamos los tres modos de lectura: `PERMISSIVE` conserva la fila con nulos, `DROPMALFORMED` la
descarta sin avisar y `FAILFAST` detiene todo (el error de la celda `modos` es intencional).
Al final, la capa **bronce**: todo como texto, sin perder ninguna fila.

In [ ]:
# [esquema_estricto]
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# El esquema que DEBERÍAN cumplir los datos (la fecha va como texto: llega en dos formatos)
ESQUEMA = StructType([
    StructField("id_venta", StringType()),
    StructField("fecha", StringType()),
    StructField("ciudad", StringType()),
    StructField("canal", StringType()),
    StructField("id_cliente", StringType()),
    StructField("id_producto", StringType()),
    StructField("cantidad", IntegerType()),
    StructField("precio_unitario", IntegerType()),
    StructField("descuento", DoubleType()),
    StructField("metodo_pago", StringType()),
])
# Una columna extra recoge, tal cual, cada fila que no cumple el esquema.
# cache() obliga a interpretar todas las columnas (lo explica la advertencia «Una trampa» del módulo)
con_registro = StructType(ESQUEMA.fields + [StructField("_corrupt_record", StringType())])
estricto = spark.read.csv("datos/ventas.csv", header=True, schema=con_registro).cache()
malas = estricto.filter(F.col("_corrupt_record").isNotNull())
print("Filas que no cumplen el esquema:", malas.count())
malas.select("id_venta", "precio_unitario", "_corrupt_record").show(3, truncate=False)

In [ ]:
# [modos]
def resumen_error(error):
    """La línea útil de un error de Spark: [CLASE_DE_ERROR] mensaje."""
    clases = re.findall(r"\[[A-Z_.]+\][^\n]*?(?=\s*SQLSTATE|: |\n|$)", str(error))
    return clases[-1].strip() if clases else str(error).splitlines()[0]


# Tres políticas ante una fila que no cumple el esquema
sc.setLogLevel("OFF")        # FAILFAST fallará a propósito: callamos el registro interno de Java
for modo in ["PERMISSIVE", "DROPMALFORMED", "FAILFAST"]:
    lectura = spark.read.csv("datos/ventas.csv", header=True, schema=ESQUEMA, mode=modo)
    try:
        filas = lectura.cache().count()      # cache() obliga a interpretar TODAS las columnas
        nulos = lectura.filter(F.col("precio_unitario").isNull()).count()
        print(f"{modo:<13} → {filas} filas · {nulos} precios nulos")
    except Exception as error:
        print(f"{modo:<13} → ERROR {resumen_error(error)}")
    lectura.unpersist()
sc.setLogLevel("ERROR")

# La trampa: sin cache(), count() no interpreta las columnas y no ve las filas malas
sin_cache = spark.read.csv("datos/ventas.csv", header=True, schema=ESQUEMA, mode="DROPMALFORMED")
print(f"{'DROPMALFORMED':<13} → {sin_cache.count()} filas, sin cache()")

In [ ]:
# [bronce]
# Capa BRONCE: todo como texto, tal como llegó. Ninguna fila se pierde ni se altera
TEXTO = StructType([StructField(campo.name, StringType()) for campo in ESQUEMA.fields])
bronce = spark.read.csv("datos/ventas.csv", header=True, schema=TEXTO)
print("Filas en bronce:", bronce.count())
bronce.printSchema()

## Bloque 2 · Módulo 11 — La limpieza de la sesión 1, en Spark
Las seis reglas de la sesión 1, ahora en PySpark. Spark 4 trabaja en **modo ANSI**: una conversión
imposible es un error, no un nulo (el error de `sp_ansi` es intencional); cuando el nulo es lo que se
quiere, se pide con las funciones `try_`. La última celda comprueba que Spark y pandas llegan a la misma tabla.

In [ ]:
# [sp_duplicados]
# UNICIDAD
v = bronce.dropDuplicates()
print("Filas:", bronce.count(), "→", v.count(), "· duplicadas:", bronce.count() - v.count())

In [ ]:
# [sp_texto]
# CONSISTENCIA: el mismo diccionario de la sesión 1, convertido en un mapa de Spark.
# create_map pide clave1, valor1, clave2, valor2…: aplanamos el diccionario en esa lista
mapa_ciudades = F.create_map(*[F.lit(x) for par in CIUDAD_CANONICA.items() for x in par])
# Normalizar como en la sesión 1: sin espacios, en minúsculas y sin tildes
norma = F.translate(F.lower(F.trim("ciudad")), "áéíóúü", "aeiouu")
sin_regla = v.filter(F.col("ciudad").isNotNull() & mapa_ciudades[norma].isNull())
print("Grafías sin regla:", sin_regla.select(norma).distinct().count())

v = (v.withColumn("ciudad", F.coalesce(mapa_ciudades[norma], F.lit("Sin dato")))
      .withColumn("canal", F.initcap(F.trim("canal"))))
v.groupBy("canal").count().orderBy("canal").show()

In [ ]:
# [sp_ansi]
# Spark 4 trabaja en modo ANSI: un dato que no encaja con la regla es un ERROR, no un nulo
dos_formatos = spark.createDataFrame([("2025-04-18",), ("18/04/2025",)], ["fecha"])
sc.setLogLevel("OFF")        # el error es intencional: callamos el registro interno de Java
try:
    dos_formatos.select(F.to_date("fecha", "yyyy-MM-dd")).show()
except Exception as error:
    print(resumen_error(error))
sc.setLogLevel("ERROR")

In [ ]:
# [sp_precios]
# VALIDEZ: la misma expresión regular de la sesión 1, y luego a número entero.
# try_cast es el errors="coerce" de Spark: lo que no se pueda convertir queda nulo y se cuenta
v = v.withColumn("precio_unitario",
                 F.regexp_replace("precio_unitario", r"[$.\s]", "").try_cast("int"))
print("Precios sin convertir:", v.filter(F.col("precio_unitario").isNull()).count())

In [ ]:
# [sp_fechas]
# VALIDEZ: cada formato de forma explícita; try_to_timestamp devuelve nulo si no encaja
iso = F.try_to_timestamp("fecha", F.lit("yyyy-MM-dd"))
dma = F.try_to_timestamp("fecha", F.lit("dd/MM/yyyy"))
v = v.withColumn("fecha", F.coalesce(iso, dma).cast("date"))
print("Fechas sin interpretar:", v.filter(F.col("fecha").isNull()).count())
v.agg(F.min("fecha").alias("desde"), F.max("fecha").alias("hasta")).show()

In [ ]:
# [sp_reglas]
# EXACTITUD: las cantidades imposibles van a cuarentena, no a la basura
v = v.withColumn("cantidad", F.col("cantidad").cast("int"))
# En Spark, comparar un nulo no da ni verdadero ni falso: sin isNull(), una cantidad vacía
# no quedaría ni en v ni en la cuarentena
fuera = F.col("cantidad").isNull() | ~F.col("cantidad").between(1, 50)
cuarentena_sp = v.filter(fuera)
v = v.filter(~fuera)
print("Filas válidas:", v.count(), "· en cuarentena:", cuarentena_sp.count())

In [ ]:
# [sp_completitud]
# COMPLETITUD: las mismas reglas de negocio documentadas en la sesión 1
v = (v.withColumn("descuento", F.col("descuento").cast("double"))
      .fillna({"descuento": 0.0, "id_cliente": "ANONIMO", "metodo_pago": "Sin dato"})
      .withColumn("total", F.col("cantidad") * F.col("precio_unitario") * (1 - F.col("descuento"))))
# Vacíos que quedan, por columna
v.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in v.columns]).show()

In [ ]:
# [sp_coherencia]
# CONTROL DE COHERENCIA: Spark y pandas deben llegar a la MISMA tabla limpia
spark_ciudad = (v.groupBy("ciudad").agg(F.sum("total").alias("total"))
                 .toPandas().set_index("ciudad")["total"])
mismas = sorted(fila.id_venta for fila in v.select("id_venta").collect()) == sorted(df["id_venta"])
print("Filas · Spark:", v.count(), "· pandas:", len(df))
print("¿Las mismas ventas?", mismas)
print("Mayor diferencia por ciudad (pesos):",
      round((spark_ciudad - df.groupby("ciudad")["total"].sum()).abs().max(), 6))

## Bloque 2 · Módulo 12 — Agregar, unir y ventanas

In [ ]:
# [sp_union]
# Unir con el catálogo. broadcast: la tabla pequeña se copia a cada ejecutor y se evita un shuffle
# Catálogo pequeño y sin errores conocidos: aquí inferir los tipos es aceptable (ver módulo 10)
productos_sp = spark.read.csv("datos/productos.csv", header=True, inferSchema=True)
ventas_sp = (v.join(F.broadcast(productos_sp.select("id_producto", "nombre", "categoria")),
                    on="id_producto", how="left")
              .withColumn("mes", F.date_format("fecha", "yyyy-MM")))
print("Ventas sin categoría:", ventas_sp.filter(F.col("categoria").isNull()).count())
ventas_sp.select("id_venta", "fecha", "ciudad", "categoria", "total").orderBy("id_venta").show(4)

In [ ]:
# [sp_ciudad]
# Dividir – aplicar – combinar, ahora en Spark
(ventas_sp.groupBy("ciudad")
          .agg(F.count("*").alias("ventas"),
               F.round(F.sum("total") / 1e6, 1).alias("millones"),
               F.round(F.avg("total")).cast("int").alias("ticket_promedio"))
          .orderBy(F.desc("millones"))
          .show())

In [ ]:
# [sp_ventana]
from pyspark.sql import Window

# Funciones de VENTANA: un cálculo por grupo sin colapsar las filas.
# Aquí, los dos productos más vendidos (en unidades) de cada categoría
unidades = ventas_sp.groupBy("categoria", "nombre").agg(F.sum("cantidad").alias("unidades"))
puesto = F.row_number().over(Window.partitionBy("categoria").orderBy(F.desc("unidades"), "nombre"))
(unidades.withColumn("puesto", puesto)
         .filter("puesto <= 2")
         .orderBy("categoria", "puesto")
         .show(truncate=False))

In [ ]:
# [sp_pivote]
# Tabla dinámica: millones de pesos por categoría y canal
(ventas_sp.groupBy("categoria")
          .pivot("canal", ["Tienda", "Web", "App"])
          .agg(F.round(F.sum("total") / 1e6, 1))
          .orderBy("categoria")
          .show())

**Ejercicio (módulo 12).** Construye una tabla con el ticket promedio (valor medio de una venta) por mes
y canal. ¿Hay algún canal que tenga siempre el ticket más alto? Resuélvelo en una celda nueva (**+ Código**); la pista y la solución están en el material HTML.

## Bloque 2 · Módulo 13 — Datos semiestructurados: los eventos de la web
JSON con campos opcionales y anidados: un nulo puede ser «no aplica» o un error de calidad.
Los eventos no siguen a una persona de la búsqueda al pago: comparan volúmenes, no son un embudo.

In [ ]:
# [logs_esquema]
# JSON por línea: Spark recorre los registros y DEDUCE un esquema que los cubra a todos
logs = spark.read.json("datos/logs_web.jsonl")
print("Eventos:", logs.count())
logs.printSchema()

In [ ]:
# [logs_campos]
# No todos los eventos traen todos los campos: count() cuenta solo los valores NO nulos
logs.select([F.count(c).alias(c) for c in logs.columns]).show()

In [ ]:
# [logs_anidados]
# Los campos anidados se leen con un punto: dispositivo.tipo, pago.medio, pago.aprobado
(logs.groupBy(F.col("dispositivo.tipo").alias("dispositivo"))
     .pivot("evento", ["buscar", "ver_producto", "agregar_carrito", "pagar"])
     .count()
     .orderBy("dispositivo")
     .show())

In [ ]:
# [logs_pagos]
pagos = logs.filter(F.col("evento") == "pagar")
(pagos.groupBy(F.col("pago.medio").alias("medio"))
      .agg(F.count("*").alias("intentos"),
           F.round(100 * F.avg(F.col("pago.aprobado").cast("int")), 1).alias("pct_aprobados"))
      .orderBy(F.desc("intentos"), "medio")
      .show())

In [ ]:
# [logs_aplanar]
# Aplanar: de JSON anidado a una tabla con columnas simples, lista para guardarla en una base de datos (sesión 3)
eventos = logs.select(
    F.to_timestamp("ts").alias("momento"),
    "sesion", "evento", "ciudad", "id_cliente", "id_producto", "consulta",
    F.col("dispositivo.tipo").alias("dispositivo"),
    F.col("dispositivo.so").alias("sistema_operativo"),
    F.col("pago.medio").alias("medio_pago"),
    F.col("pago.aprobado").alias("pago_aprobado"),
    "duracion_ms",
)
eventos.orderBy("momento", "sesion").show(3)

**Ejercicio (módulo 13).** ¿Qué buscan más los clientes en la web y la app? Muestra las cinco búsquedas
más frecuentes, con su reparto entre móvil y escritorio. Resuélvelo en una celda nueva (**+ Código**); la pista y la solución están en el material HTML.

## Para el taller
El enunciado completo y la rúbrica están en el módulo 14 del material HTML.
Sigue trabajando en este mismo notebook: ya tienes HDFS en marcha, la sesión de Spark,
la tabla plata `v`, las ventas unidas al catálogo `ventas_sp` y los eventos web, crudos (`logs`)
y aplanados (`eventos`).

Al terminar, puedes apagar los servicios con la celda siguiente. No la ejecutes antes de acabar:
`spark.stop()` deja inservibles los DataFrames y borra las vistas temporales, y con HDFS detenido
no corren los jobs de Hadoop: habría que volver a ejecutar el notebook.

In [ ]:
# [apagar]
# Apagar Spark y los servicios de HDFS (Colab también los apaga al cerrar el entorno)
spark.stop()
!hdfs --daemon stop datanode
!hdfs --daemon stop namenode
print("Servicios detenidos")